# AmSC Data Catalog Explorer

This notebook explores the AmSC Data Catalog on the **staging** environment.
It discovers available catalogs, lists datasets, and inspects entity details.

**What you'll do:**
1. Connect to the staging AmSC API
2. Discover available catalogs
3. Browse datasets within each catalog
4. Inspect individual entity details
5. Search by keyword

**Prerequisites:**
- Install dependencies: `pip install -r requirements.txt`
- A [Globus](https://www.globus.org/) account

**Authentication:**
A browser window will open for Globus login on first API call.
Tokens are cached locally for subsequent runs.

In [ ]:
import os
from amsc_client import Client, ApiError

## Step 1: Connect to the Staging API

In [ ]:
base_url='https://api.american-science-cloud.org/api/current'
globus_app_id='e4f48665-38b5-4833-a89e-849c71f5b3e3' # ChildersAmSC ->  AmSC CLI Client
client = Client(
    base_url=base_url,
    auth_method="globus",
    globus_client_id=globus_app_id,
    requested_scopes=f'openid profile email https://auth.globus.org/scopes/{globus_app_id}/amsc_test',
    resource_server=globus_app_id,
    use_id_token=True,
)
print(f"✅ Connected to {base_url}")

## Step 2: Discover Available Catalogs

We search for all entries and extract the unique catalog names from
their fully qualified names (FQNs). An FQN looks like:
```
catalog-storage.catalog-name.scientific-work.artifact
```

In [ ]:
# Search for everything
results = client.catalog.search("*", limit=50)
print(f"Total entities in catalog: {results.total_count}")

# Extract unique catalogs from FQNs
catalogs = {}
for item in results:
    parts = item.fqn.split(".")
    if len(parts) >= 2:
        catalog_name = ".".join(parts[:2])
        if catalog_name not in catalogs:
            catalogs[catalog_name] = {"scientificWork": 0, "artifact": 0, "other": 0}
        t = item.type or "other"
        if t in catalogs[catalog_name]:
            catalogs[catalog_name][t] += 1
        else:
            catalogs[catalog_name]["other"] += 1

print(f"\nFound {len(catalogs)} catalog(s):\n")
for name, counts in sorted(catalogs.items()):
    print(f"📁 {name}")
    print(f"   ScientificWorks: {counts['scientificWork']}")
    print(f"   Artifacts:       {counts['artifact']}")
    if counts['other']:
        print(f"   Other:           {counts['other']}")

## Step 3: Browse Datasets

List the first 10 ScientificWorks (research projects) from each catalog,
along with their artifacts (datasets).

In [ ]:
# List ScientificWorks
results = client.catalog.search("*", limit=50)

# Group by type
works = [r for r in results if r.type == "scientificWork"]
artifacts = [r for r in results if r.type == "artifact"]

print(f"ScientificWorks (showing first 10 of {len(works)}):\n")
for i, work in enumerate(works[:10], 1):
    # Get the short name from the FQN
    short_name = work.fqn.split(".")[-1] if work.fqn else work.name
    desc = (work.description or "")[:80]
    print(f"  {i:2d}. {short_name}")
    if desc:
        print(f"      {desc}...")

print(f"\nArtifacts (showing first 10 of {len(artifacts)}):\n")
for i, art in enumerate(artifacts[:10], 1):
    short_name = art.fqn.split(".")[-1] if art.fqn else art.name
    parent = ".".join(art.fqn.split(".")[:-1]) if art.fqn else "unknown"
    print(f"  {i:2d}. {short_name}")
    print(f"      Parent: {parent}")

## Step 4: Inspect Entity Details

Use `client.catalog.get(fqn)` to fetch the full metadata for an entity.
The returned object is a typed wrapper (`Artifact`, `ScientificWork`, etc.)
with all fields accessible as properties.

In [ ]:
# Pick the first ScientificWork and its artifacts
if works:
    work_fqn = works[0].fqn
    print(f"Inspecting: {work_fqn}\n")

    try:
        entity = client.catalog.get(work_fqn)
        print(f"  Name:        {entity.name}")
        print(f"  Type:        {entity.type}")
        print(f"  Description: {(entity.description or 'N/A')[:120]}")
        print(f"  Location:    {getattr(entity, 'location', 'N/A')}")
        print(f"  FQN:         {entity.fqn}")
    except ApiError as e:
        print(f"  ❌ Failed: {e}")
else:
    print("No ScientificWorks found")

In [ ]:
# Inspect the first few artifacts in detail
print(f"Artifact details (first 5):\n")
for art in artifacts[:5]:
    try:
        entity = client.catalog.get(art.fqn)
        print(f"  📄 {entity.name}")
        print(f"     Type:        {entity.type}")
        print(f"     Description: {(entity.description or 'N/A')[:100]}")
        print(f"     Location:    {getattr(entity, 'location', 'N/A')}")
        print(f"     Format:      {getattr(entity, 'format', 'N/A')}")
        print(f"     Size:        {getattr(entity, 'size', 'N/A')}")
        print()
    except ApiError as e:
        print(f"  ❌ Failed to get {art.fqn}: {e}\n")

## Step 5: Search by Keyword

Search for specific topics in the catalog. Try different keywords
to explore the available data.

In [ ]:
SEARCH_TERM = "simulation"  # Change this to explore different topics

print(f"Searching for '{SEARCH_TERM}'...\n")
try:
    results = client.catalog.search(SEARCH_TERM, limit=10)
    print(f"Found {results.total_count} total matches (showing {len(results)}):\n")
    for i, item in enumerate(results, 1):
        print(f"  {i}. {item.name} ({item.type})")
        if item.description:
            print(f"     {item.description[:100]}")
        print(f"     FQN: {item.fqn}")
        print()
except ApiError as e:
    print(f"Search failed: {e}")

In [ ]:
# Try a few more searches
for term in ["molecular", "climate", "benchmark", "energy"]:
    try:
        results = client.catalog.search(term, limit=5)
        count = results.total_count
        print(f"  '{term}': {count} result(s)")
        for item in results:
            print(f"    - {item.name} ({item.type})")
    except ApiError:
        print(f"  '{term}': search failed")
    print()

## Summary

| Operation | API Call | Returns |
|-----------|---------|----------|
| Search | `client.catalog.search(query, limit=N)` | `SearchResults` (iterable) |
| Get entity | `client.catalog.get(fqn)` | `Artifact`, `ScientificWork`, etc. |

**SearchResults API:**
- `len(results)` — items in this page
- `results.total_count` — total matches across all pages
- `results[i]` — access by index
- `for item in results` — iterate

**Entity properties:**
- `entity.name`, `entity.fqn`, `entity.description`
- `entity.type` (friendly alias for `type_`)
- `entity.format` (friendly alias for `format_`)
- `entity.location`, `entity.size`

Everything is imported from `amsc_client` — no need to import from `amsc_api_autogen`.